In [ ]:
# 1. データ準備とスコア算出
import pandas as pd
import numpy as np

# データの読み込み
df_players = pd.read_csv('/Users/akihirookuyama/Soccer_Score_App/Data/RawData/df_player_stats.csv')

# 欠損値の処理
fill_cols = ['goals', 'assists', 'cbp_attack', 'cbp_attack_90', 
             'shoot_point', 'shoot_point_90', 'goal_point', 'goal_point_90']
df_players[fill_cols] = df_players[fill_cols].fillna(0)

# Goal Propensity Score (GPS) の算出
# 重み設定
WEIGHT_GOAL_PT = 1.0
WEIGHT_SHOOT_PT = 0.5
WEIGHT_CBP_ATTACK = 0.2

df_players['Goal Propensity Score'] = (
    (df_players['goal_point_90'] * WEIGHT_GOAL_PT) +
    (df_players['shoot_point_90'] * WEIGHT_SHOOT_PT) +
    (df_players['cbp_attack_90'] * WEIGHT_CBP_ATTACK)
)

# 2025年の名古屋グランパスを抽出して降順にソート
nagoya_2025 = df_players[(df_players['year'] == 2025) & (df_players['team_name'] == '名古屋グランパス')].copy()
nagoya_2025_sorted = nagoya_2025.sort_values(by='Goal Propensity Score', ascending=False)

# 表示
cols_to_show = ['player_name', 'pos', 'games', 'goals', 'goal_point_90', 'shoot_point_90', 'cbp_attack_90', 'Goal Propensity Score']
print(nagoya_2025_sorted[cols_to_show].head(15))

     player_name pos  games  goals  goal_point_90  shoot_point_90  \
1967   マテウス　カストロ  FW     25      5           0.84            2.16   
1966        稲垣　祥  MF     38     11           0.53            0.91   
1969       山岸　祐也  FW     28      3           0.59            0.99   
1979       永井　謙佑  FW     36      2           0.22            1.06   
1970       和泉　竜司  MF     37      4           0.34            0.84   
1978       木村　勇大  FW     14      1           0.00            1.91   
1976       佐藤　瑶大  DF     27      5           0.34            0.49   
1977       菊地　泰智  MF     20      1           0.78            0.00   
1971        原　輝綺  DF     29      3           0.50            0.00   
1972       椎橋　慧也  MF     28      2           0.19            0.40   
1973       徳元　悠平  DF     24      2           0.32            0.00   
1975       浅野　雄也  MF     25      0           0.00            1.09   
1968        森島　司  MF     36      1           0.00            0.47   
1974       中山　克広  MF     26      0

In [ ]:
# 2. 手法の比較検証（決定論 vs モンテカルロ）
# 占有率P_iが高い順にN名を抽出する方法と占有率P_iを確率分布として扱い、ランダムにN回の抽選を実施するモンテカルロ法を比較する
def verify_methods(team_name, year):
    # 特定チーム・年度の抽出
    team_df = df_players[(df_players['team_name'] == team_name) & (df_players['year'] == year)].copy()
    actual_total_goals = int(team_df['goals'].sum())
    
    if actual_total_goals == 0:
        return "対象データに得点がありません。"

    # --- A. 期待値上位方式 (Deterministic) ---
    det_results = pd.Series(0, index=team_df['player_name'])
    top_player = team_df.sort_values('Goal Propensity Score', ascending=False).iloc[0]['player_name']
    det_results[top_player] = actual_total_goals

    # --- B. モンテカルロサンプリング (Monte Carlo) ---
    mc_results = pd.Series(0, index=team_df['player_name'])
    total_score = team_df['Goal Propensity Score'].sum()
    
    if total_score > 0:
        probs = team_df['Goal Propensity Score'] / total_score
        sampled = np.random.choice(team_df['player_name'], size=actual_total_goals, p=probs)
        for p in sampled:
            mc_results[p] += 1

    # 結果の統合
    comparison = pd.DataFrame({
        'Actual': team_df.set_index('player_name')['goals'],
        'Deterministic': det_results,
        'Monte_Carlo': mc_results
    }).sort_values('Actual', ascending=False)

    # RMSEの計算
    rmse_det = np.sqrt(((comparison['Actual'] - comparison['Deterministic'])**2).mean())
    rmse_mc = np.sqrt(((comparison['Actual'] - comparison['Monte_Carlo'])**2).mean())
    
    print(f"--- {year} {team_name} 検証結果 ---")
    print(f"期待値上位方式の誤差 (RMSE): {rmse_det:.3f}")
    print(f"モンテカルロ方式の誤差 (RMSE): {rmse_mc:.3f}")
    
    return comparison.head(10)

# 実行
comparison_df = verify_methods('名古屋グランパス', 2024)
display(comparison_df)

--- 2024 名古屋グランパス 検証結果 ---
期待値上位方式の誤差 (RMSE): 10.066
モンテカルロ方式の誤差 (RMSE): 2.033


,Actual,Deterministic,Monte_Carlo
player_name,,,
稲垣 祥,6,0,2
永井 謙佑,6,0,5
パトリック,5,0,4
キャスパー ユンカー,4,0,7
森島 司,3,0,1
和泉 竜司,3,0,3
ハ チャンレ,3,0,2
山岸 祐也,2,0,3
三國 ケネディエブス,2,0,0



モンテカルロ・サンプリングを採用する

理由：
- 「外れ」を予測できる: サッカーは、シュート1本のGPSが0.05でも入る時は入ります。確率手法なら、その「0.05」を引く可能性を排除しません。
- エンターテインメント性: 毎回同じ選手が予測されるよりも、毎回微妙に異なる「可能性のある得点者リスト」が出る方が、ユーザーにとっての納得感と楽しみが増します。
- 複数得点への対応: チーム予測が「3点」となった際、エースが2点、ボランチが1点、といった「ありそうなスコアライン」を生成できます。

In [ ]:
# 3. 【最終成果物】得点者予測シミュレーター関数
def predict_goalscorers(team_name, year, predicted_goals):
    """
    指定したチーム・年度の Goal Propensity Score に基づき、
    予測された得点数分の得点者をシミュレートする
    """
    if predicted_goals <= 0:
        return []

    # 対象チームの全選手データを取得
    team_data = df_players[(df_players['team_name'] == team_name) & (df_players['year'] == year)]
    
    if team_data.empty:
        return ["選手データなし"] * int(predicted_goals)

    # スコアの合計を計算
    total_score = team_data['Goal Propensity Score'].sum()
    
    if total_score == 0:
        probs = [1/len(team_data)] * len(team_data)
    else:
        probs = team_data['Goal Propensity Score'] / total_score

    # モンテカルロサンプリング実行
    scorers = np.random.choice(team_data['player_name'], size=int(predicted_goals), p=probs)
    
    return list(scorers)

# --- 使用例 ---
team = '名古屋グランパス'
target_year = 2025
goals = 2

result = predict_goalscorers(team, target_year, goals)
print(f"\n【予測結果】 {team} ({target_year}) が {goals} 点取った場合の得点者:")
print(f"-> {', '.join(result)}")


【予測結果】 名古屋グランパス (2025) が 2 点取った場合の得点者:
-> 森島　司, マテウス　カストロ
